> **Public release note.** Notebook outputs and execution counts have been removed because the underlying Malaysian Motor claims data are confidential. Local user-specific paths and record identifiers have also been removed. The code documents the analysis workflow but cannot be executed end-to-end without appropriately structured confidential input data.


> **Terminology note.** The later individual-claims source was internally labelled `V11`; some variable and processed-file names retain that label for audit continuity. In the dissertation it is referred to as the later BI claims extract.


# Later BI claims extract — 2022 Q4 to 2026 Q1 out-of-time validation

This notebook freezes the valuation date at 2022 Q4 and tests the previously fitted
Random Forest and XGBoost models against BI Excess development observed through
2026 Q1. It separately estimates pure IBNR using the same historical frequency--severity
framework and compares the completed Integrated Framework with an incurred Chain
Ladder benchmark.

## 0. Run order

Run Notebook 08 first so that the restated 2022 Q4 bridge and emergence decomposition
are available. This notebook should then be run from top to bottom; the fitted RF and
XGBoost artefacts are loaded but are not retrained or recalibrated using the later outcomes.

In [ ]:
from pathlib import Path
import gc
import json
import pickle
import re
import warnings

import numpy as np
import pandas as pd
import polars as pl
import pyarrow.parquet as pq

try:
    import joblib
except Exception:
    joblib = None

warnings.filterwarnings("ignore", category=FutureWarning)

print("Polars:", pl.__version__)
print("Pandas:", pd.__version__)

## 1. Define the project and input paths

 
The  cell defines the file  location, together with the original master data, later
claims extract, Notebook 08 outputs and historical back-test files.

In [ ]:
PROJECT_DIR = Path("/path/to/BI_large_claims_project")
PROCESSED_DIR = PROJECT_DIR / "processed"

OLD_MASTER_PARQUET = PROJECT_DIR / "BI_large_claims_master_long.parquet"
V11_PARQUET = PROCESSED_DIR / "BI_LARGE_CLAIMS_IFOA_REPAIRED.parquet"

BRIDGE_CSV = PROCESSED_DIR / "V11_2022Q4_to_latest_BIXS_validation_bridge.csv"
EMERGENCE_AY_CSV = (
    PROCESSED_DIR
    / "V11_2022Q4_to_latest_BIXS_emergence_decomposition_by_AY.csv"
)
EMERGENCE_PORTFOLIO_CSV = (
    PROCESSED_DIR
    / "V11_2022Q4_to_latest_BIXS_emergence_decomposition_portfolio.csv"
)

HIST_1YR = (
    PROCESSED_DIR
    / "chapter5_outputs"
    / "section_5_9A_historical_valuation_diagonal"
    / "historical_diagonal_claim_level_predictions.parquet"
)

HIST_2YR = (
    PROCESSED_DIR
    / "chapter5_outputs"
    / "section_5_9C_historical_valuation_diagonal_2yr"
    / "historical_diagonal_claim_level_predictions.parquet"
)

BASE_VALIDATION_DIR = PROCESSED_DIR / "out_of_time_2022q4_validation"
OUT_DIR = PROCESSED_DIR / "out_of_time_2022q4_validation_consistent_pure_ibnr"
OUT_DIR.mkdir(parents=True, exist_ok=True)

required_inputs = {
    "Original master claims history": OLD_MASTER_PARQUET,
    "Validated later claims extract": V11_PARQUET,
    "2022 Q4 validation bridge": BRIDGE_CSV,
    "Emergence by accident year": EMERGENCE_AY_CSV,
    "Emergence portfolio": EMERGENCE_PORTFOLIO_CSV,
    "One-year historical deployment": HIST_1YR,
    "Two-year historical deployment": HIST_2YR,
}

missing_inputs = [
    f"{label}: {path}"
    for label, path in required_inputs.items()
    if not path.exists()
]

if missing_inputs:
    raise FileNotFoundError(
        "Required input files are missing:\n" + "\n".join(missing_inputs)
    )

print("Project folder:", PROJECT_DIR)
print("Output folder:", OUT_DIR)

## 2. Configuration

The 2022 Q4 claim snapshot can be rebuilt if required, while the pure IBNR estimate uses
three historical comparator accident years. The later outcomes are used only for validation;
they are not used to fit, tune or recalibrate the frozen machine-learning models.

In [ ]:
BUILD_SNAPSHOT = True
PURE_IBNR_ROLLING_COMPARATOR_YEARS = 3

print("Output directory:", OUT_DIR)

## 3. Validate the observed out-of-time outcome

The bridge created in Notebook 08 is loaded and checked before any model scoring begins.
This fixes the restated 2022 Q4 starting position and the BI Excess development subsequently
observed through 2026 Q1, so all methods are compared with the same realised outcome.

In [ ]:
if not BRIDGE_CSV.exists():
    raise FileNotFoundError(
        "The saved validation bridge was not found. "
        "Run the V11 reconciliation notebook first."
    )

bridge = pl.read_csv(BRIDGE_CSV)
display(bridge)

if EMERGENCE_AY_CSV.exists():
    emergence_ay = pl.read_csv(EMERGENCE_AY_CSV)
    display(emergence_ay)
else:
    emergence_ay = None
    print("AY decomposition CSV not found.")

if EMERGENCE_PORTFOLIO_CSV.exists():
    emergence_portfolio = pl.read_csv(EMERGENCE_PORTFOLIO_CSV)
    display(emergence_portfolio)
else:
    emergence_portfolio = None
    print("Portfolio decomposition CSV not found.")

In [ ]:
portfolio_bridge = bridge.select(
    [
        pl.col("old_bixs_2022q4").sum().alias("old_bixs_2022q4"),
        pl.col("v11_bixs_2022q4").sum().alias("v11_bixs_2022q4"),
        pl.col("v11_latest_bixs").sum().alias("v11_latest_bixs"),
        pl.col("bixs_restatement").sum().alias("total_restatement"),
        pl.col("bixs_post_2022_emergence").sum().alias("actual_total_post_2022"),
    ]
)

display(portfolio_bridge)

ACTUAL_TOTAL_POST_2022 = float(
    portfolio_bridge["actual_total_post_2022"][0]
)

if emergence_portfolio is not None:
    actual_reported = float(
        emergence_portfolio
        .filter(pl.col("EMERGENCE_TYPE").is_in([
            "BIXS_AT_2022Q4",
            "REPORTED_BIC_TO_BIXS",
        ]))
        ["bixs_development"]
        .sum()
    )

    actual_pure_ibnr = float(
        emergence_portfolio
        .filter(pl.col("EMERGENCE_TYPE") == "PURE_IBNR_PROXY")
        ["bixs_development"]
        .sum()
    )

    print(f"Actual reported-claim emergence: MYR {actual_reported/1e6:,.2f}m")
    print(f"Actual pure-IBNR proxy:          MYR {actual_pure_ibnr/1e6:,.2f}m")
    print(f"Actual total emergence:          MYR {ACTUAL_TOTAL_POST_2022/1e6:,.2f}m")
    print(
        "Reconciliation difference:",
        round((actual_reported + actual_pure_ibnr) - ACTUAL_TOTAL_POST_2022, 6)
    )

## 4. Recover the exact 2022 Q4 valuation diagonal

The original master data determine the development quarter available for each accident
quarter at 2022 Q4. Reusing that exact diagonal prevents the later extract from giving the
models additional maturity that would not have been available at the valuation date.

In [ ]:
old_diagonal = (
    pl.scan_parquet(OLD_MASTER_PARQUET)
    .group_by(["ACC_YEAR", "ACC_QTR"])
    .agg(pl.col("DEV_QTR").max().alias("T0_DEV_QTR"))
    .sort(["ACC_YEAR", "ACC_QTR"])
    .collect()
)

assert old_diagonal.height == 52, old_diagonal.height

old_diagonal_map = {
    (int(r["ACC_YEAR"]), int(r["ACC_QTR"])): int(r["T0_DEV_QTR"])
    for r in old_diagonal.to_dicts()
}

display(old_diagonal.head(8))
display(old_diagonal.tail(8))

assert old_diagonal_map[(2010, 1)] == 51
assert old_diagonal_map[(2022, 4)] == 0

print("Confirmed valuation date: 2022 Q4")

## 5. Build the later-extract claim snapshot at 2022 Q4

The later claims history is cut back to the reconstructed 2022 Q4 diagonal and reduced to
one row per claim at that date. The resulting snapshot uses the restated 2022 Q4 financial
position while preserving the historical information set required for out-of-time scoring.

In [ ]:
T0_SNAPSHOT_PARQUET = BASE_VALIDATION_DIR / "v11_2022q4_claim_snapshot.parquet"

def build_t0_snapshot():
    if T0_SNAPSHOT_PARQUET.exists():
        print("Using existing:", T0_SNAPSHOT_PARQUET)
        return pl.read_parquet(T0_SNAPSHOT_PARQUET)

    parts = []

    for year in range(2010, 2023):
        print("Building T0 snapshot AY", year)

        lf = pl.scan_parquet(V11_PARQUET)

        q_parts = []
        for qtr in [1, 2, 3, 4]:
            dev = old_diagonal_map[(year, qtr)]

            q = (
                lf
                .filter(
                    (pl.col("ACC_YEAR") == year)
                    & (pl.col("ACC_QTR") == qtr)
                    & (pl.col("DEV_QTR") == dev)
                )
                .collect()
            )
            q_parts.append(q)

        parts.append(pl.concat(q_parts))
        gc.collect()

    out = pl.concat(parts)

    key_cols = [
        "CLASS", "COVER", "NATLOSS",
        "ACC_YEAR", "ACC_QTR", "CLAIMS_KEY"
    ]

    dupes = (
        out
        .group_by(key_cols)
        .len()
        .filter(pl.col("len") > 1)
    )

    if dupes.height:
        raise RuntimeError(
            f"Unexpected duplicate T0 claim identities: {dupes.height}"
        )

    out.write_parquet(T0_SNAPSHOT_PARQUET, compression="zstd")
    print("Saved:", T0_SNAPSHOT_PARQUET)
    return out

if BUILD_SNAPSHOT:
    t0 = build_t0_snapshot()
    print("T0 rows:", t0.height)
    print("T0 accident years:", t0["ACC_YEAR"].min(), "to", t0["ACC_YEAR"].max())
    print("T0 DEV_QTR:", t0["DEV_QTR"].min(), "to", t0["DEV_QTR"].max())
else:
    t0 = pl.read_parquet(T0_SNAPSHOT_PARQUET)

display(t0.head())

## 6. Reconstruct the helper fields required by the frozen models

Some model inputs are cumulative or history-based fields rather than values stored directly
on a single 2022 Q4 row. This section rebuilds those fields from each claim's history up to
$T_0$ only, so the later extract can be passed through the original fitted preprocessing
pipeline without using post-valuation information.

In [ ]:
# Historical reference diagnostics retained for auditability.

ref = pl.read_parquet(
    HIST_1YR,
    columns=[
        "CUM_PAIDLS",
        "CUM_INC_AMT",
        "BALOS",
        "CUM_INC_LARGE",
        "CLAIMS_CNT",
        "SETTLED_CNT",
        "CLAIMS_CNT_LARGE",
        "SETTLED_CNT_LARGE",
    ],
)

historical_rule_checks = ref.select(
    [
        (
            pl.col("CLAIMS_CNT_LARGE")
            ==
            (pl.col("CUM_INC_LARGE").fill_null(0) > 0).cast(pl.Int64)
        )
        .mean()
        .alias("large_count_rule_match"),

        (
            pl.col("SETTLED_CNT_LARGE")
            ==
            (
                (pl.col("CUM_INC_LARGE").fill_null(0) > 0)
                & (pl.col("BALOS").fill_null(0) == 0)
            ).cast(pl.Int64)
        )
        .mean()
        .alias("large_settled_rule_match"),

        (
            pl.col("SETTLED_CNT")
            ==
            (
                (pl.col("CLAIMS_CNT") == 1)
                & (pl.col("BALOS").fill_null(0) == 0)
                & (
                    (pl.col("CUM_PAIDLS").fill_null(0) != 0)
                    | (pl.col("CUM_INC_AMT").fill_null(0) != 0)
                )
            ).cast(pl.Int64)
        )
        .mean()
        .alias("ordinary_settled_rule_match"),
    ]
)

display(historical_rule_checks)

In [ ]:
# Build CLAIMS_CNT from each V11 claim's complete observed history up to
# the exact 2022 Q4 valuation diagonal. This avoids using post-valuation
# information and avoids assuming that a zero current balance means
# the claim was never reported.

claim_id = [
    "CLASS", "COVER", "NATLOSS",
    "ACC_YEAR", "ACC_QTR", "CLAIMS_KEY"
]

history_flag_parts = []

for year in range(2010, 2023):
    print(f"Reconstructing CLAIMS_CNT history for AY {year}...")

    q_conditions = []

    for qtr in [1, 2, 3, 4]:
        t0_dev = old_diagonal_map[(year, qtr)]

        q_conditions.append(
            (
                (pl.col("ACC_QTR") == qtr)
                & (pl.col("DEV_QTR") <= t0_dev)
            )
        )

    within_t0 = q_conditions[0]
    for q_cond in q_conditions[1:]:
        within_t0 = within_t0 | q_cond

    flags_y = (
        pl.scan_parquet(V11_PARQUET)
        .filter(
            (pl.col("ACC_YEAR") == year)
            & within_t0
        )
        .select(
            claim_id
            + [
                "CUM_PAIDLS",
                "CUM_INC_AMT",
                "BALOS",
            ]
        )
        .with_columns(
            (
                (pl.col("CUM_PAIDLS").fill_null(0) != 0)
                | (pl.col("CUM_INC_AMT").fill_null(0) != 0)
                | (pl.col("BALOS").fill_null(0) != 0)
            ).alias("_EVER_FINANCIAL_SIGNAL")
        )
        .group_by(claim_id)
        .agg(
            pl.col("_EVER_FINANCIAL_SIGNAL")
            .any()
            .cast(pl.Int64)
            .alias("CLAIMS_CNT")
        )
        .collect()
    )

    history_flag_parts.append(flags_y)
    gc.collect()

history_flags = pl.concat(history_flag_parts)

print("History flags:", history_flags.height)
print("T0 rows:", t0.height)

display(
    history_flags
    .group_by("CLAIMS_CNT")
    .len()
    .sort("CLAIMS_CNT")
)

In [ ]:
# Join the history-based CLAIMS_CNT to the saved T0 snapshot and reconstruct
# the remaining helper variables.

t0_model = (
    t0
    .join(
        history_flags,
        on=claim_id,
        how="left",
    )
    .with_columns(
        [
            pl.col("CLAIMS_CNT").fill_null(0).cast(pl.Int64),

            (
                (pl.col("CUM_PAIDLS").fill_null(0) != 0)
                | (pl.col("CUM_INC_AMT").fill_null(0) != 0)
                | (pl.col("BALOS").fill_null(0) != 0)
            ).alias("_CURRENT_FINANCIAL_SIGNAL"),
        ]
    )
    .with_columns(
        [
            (
                (pl.col("CLAIMS_CNT") == 1)
                & (pl.col("BALOS").fill_null(0) == 0)
                & pl.col("_CURRENT_FINANCIAL_SIGNAL")
            )
            .cast(pl.Int64)
            .alias("SETTLED_CNT"),

            (
                pl.col("CUM_INC_LARGE").fill_null(0) > 0
            )
            .cast(pl.Int64)
            .alias("CLAIMS_CNT_LARGE"),

            (
                (pl.col("CUM_INC_LARGE").fill_null(0) > 0)
                & (pl.col("BALOS").fill_null(0) == 0)
            )
            .cast(pl.Int64)
            .alias("SETTLED_CNT_LARGE"),
        ]
    )
    .drop("_CURRENT_FINANCIAL_SIGNAL")
    .with_columns(
        [
            pl.col("COVER").alias("cover"),
            pl.col("ACC_QTR").alias("ACC_QTR_feature"),
            pl.col("ACC_YEAR").alias("ACC_YEAR_feature"),
            pl.col("CLAIMS_CNT").alias("CLAIMS_CNT_feature"),
            pl.col("CUM_INC_LARGE")
            .fill_null(0)
            .alias("CUM_INC_LARGE_feature"),
        ]
    )
)

missing_claim_count = t0_model["CLAIMS_CNT"].null_count()
assert missing_claim_count == 0

print("Model-ready T0 rows:", t0_model.height)
print("Model-ready T0 columns:", len(t0_model.columns))

display(
    t0_model
    .select(
        [
            "CLAIMS_CNT",
            "SETTLED_CNT",
            "CLAIMS_CNT_LARGE",
            "SETTLED_CNT_LARGE",
        ]
    )
    .unpivot()
    .group_by(["variable", "value"])
    .len()
    .sort(["variable", "value"])
)

### Validate the BI Excess-at-snapshot flag

The archived BI Excess indicator is checked against the reconstructed 2022 Q4 financial
position and then added to the model input. Any disagreement would indicate that the later
snapshot does not reproduce the original model definition correctly.

In [ ]:
# Validate the archived definition first.
bixs_flag_ref = pl.read_parquet(
    HIST_1YR,
    columns=[
        "CUM_INC_LARGE",
        "is_bi_excess_at_snapshot",
    ],
)

bixs_flag_match = float(
    (
        bixs_flag_ref["is_bi_excess_at_snapshot"]
        ==
        (bixs_flag_ref["CUM_INC_LARGE"].fill_null(0) > 0).cast(pl.Int64)
    ).mean()
)

print("Historical is_bi_excess_at_snapshot rule match:", bixs_flag_match)

if bixs_flag_match < 0.99999:
    raise RuntimeError(
        "Archived is_bi_excess_at_snapshot does not match CUM_INC_LARGE > 0 "
        "closely enough to reconstruct safely."
    )

t0_model = t0_model.with_columns(
    (
        pl.col("CUM_INC_LARGE").fill_null(0) > 0
    )
    .cast(pl.Int64)
    .alias("is_bi_excess_at_snapshot")
)

print(
    "Added is_bi_excess_at_snapshot. Positive flags:",
    int(t0_model["is_bi_excess_at_snapshot"].sum()),
)

## 7. Recover the original maturity-based deployment rule

The historical back-tests are used to reproduce the same deployment bands applied in the
dissertation: DEV4 for development quarters 4--7, DEV8 for 8--11, DEV12 for 12--15,
and carry-forward for more mature claims. Claims below DEV_QTR_4 remain outside the
claim-level model scope.

In [ ]:
band_ref = (
    pl.scan_parquet(HIST_1YR)
    .filter(pl.col("included_in_main_diagonal") == True)
    .group_by(["DEV_QTR", "model_band"])
    .len()
    .sort(["DEV_QTR", "len"], descending=[False, True])
    .collect()
)

band_mode = (
    band_ref
    .group_by("DEV_QTR", maintain_order=True)
    .first()
    .select(["DEV_QTR", "model_band"])
    .sort("DEV_QTR")
)

display(band_mode)

band_lookup = {
    int(r["DEV_QTR"]): str(r["model_band"])
    for r in band_mode.to_dicts()
}

ref_max_dev = max(band_lookup)
ref_top_band = band_lookup[ref_max_dev]

print("Reference DEV range:", min(band_lookup), "to", ref_max_dev)
print("Top reference band:", ref_top_band)

def assign_model_band(dev):
    dev = int(dev)

    if dev in band_lookup:
        return band_lookup[dev]

    if dev < min(band_lookup):
        return "UNSUPPORTED_EARLY"

    if dev > ref_max_dev and "MATURE" in ref_top_band.upper():
        return ref_top_band

    return "UNMAPPED"

t0_model = t0_model.with_columns(
    pl.col("DEV_QTR")
    .map_elements(assign_model_band, return_dtype=pl.String)
    .alias("model_band")
)

band_summary = (
    t0_model
    .group_by(["model_band", "DEV_QTR"])
    .len()
    .sort(["model_band", "DEV_QTR"])
)

display(band_summary)

if "UNMAPPED" in set(t0_model["model_band"].unique().to_list()):
    raise RuntimeError(
        "Some development quarters could not be mapped to the archived deployment rule."
    )

## 8. Build the reported-claim validation population

The 2022 Q4 snapshot is joined to the later observed claim position and restricted to claims
that were already reported at $T_0$ and fall within a supported maturity band. This creates
the population on which the frozen RF and XGBoost reported-claim predictions can be
compared directly with later observed development.

In [ ]:
VALIDATION_POP_PARQUET = (
    BASE_VALIDATION_DIR / "v11_2022q4_reported_claim_validation_population.parquet"
)

claim_id = [
    "CLASS", "COVER", "NATLOSS",
    "ACC_YEAR", "ACC_QTR", "CLAIMS_KEY"
]

REQUIRED_VALIDATION_FEATURES = [
    "CLASS",
    "cover",
    "NATLOSS",
    "ACC_QTR",
    "ACC_YEAR",
    "PAIDLS",
    "BALOS",
    "TAG_INC_LARGE",
    "TAG_PAIDLS_LARGE",
    "CLAIMS_CNT",
    "SETTLED_CNT",
    "CUM_PAIDLS",
    "CUM_INC_AMT",
    "CUM_INC_NON_LARGE",
    "CUM_INC_LARGE",
    "CLAIMS_CNT_LARGE",
    "SETTLED_CNT_LARGE",
    "CUM_PAIDLS_NON_LARGE",
    "CUM_PAIDLS_LARGE",
    "is_bi_excess_at_snapshot",
    "model_band",
    "t0_bixs_v11",
    "latest_bixs_v11",
    "actual_future_bixs",
]


def latest_v11_for_year(year):
    lf = (
        pl.scan_parquet(V11_PARQUET)
        .filter(pl.col("ACC_YEAR") == year)
        .select(
            claim_id
            + [
                "DEV_QTR",
                "CUM_INC_LARGE",
            ]
        )
    )

    df = lf.collect()

    latest = (
        df
        .sort(claim_id + ["DEV_QTR"])
        .group_by(claim_id, maintain_order=True)
        .last()
        .select(
            claim_id
            + [
                pl.col("CUM_INC_LARGE")
                .fill_null(0)
                .alias("latest_bixs_v11")
            ]
        )
    )

    del df
    gc.collect()
    return latest


def build_validation_population():
    parts = []

    for year in range(2010, 2023):
        print("Joining latest V11 outcome for AY", year)

        latest = latest_v11_for_year(year)
        t0_y = t0_model.filter(pl.col("ACC_YEAR") == year)

        joined = (
            t0_y
            .join(latest, on=claim_id, how="left")
            .with_columns(
                [
                    pl.col("CUM_INC_LARGE")
                    .fill_null(0)
                    .alias("t0_bixs_v11"),

                    pl.col("latest_bixs_v11")
                    .fill_null(0),
                ]
            )
            .with_columns(
                [
                    (
                        pl.col("latest_bixs_v11")
                        - pl.col("t0_bixs_v11")
                    ).alias("actual_future_bixs"),

                    (
                        pl.col("t0_bixs_v11") > 0
                    )
                    .cast(pl.Int64)
                    .alias("is_bi_excess_at_snapshot"),
                ]
            )
        )

        parts.append(joined)

        del latest, joined, t0_y
        gc.collect()

    out = pl.concat(parts)

    missing = [
        c for c in REQUIRED_VALIDATION_FEATURES
        if c not in out.columns
    ]

    if missing:
        raise RuntimeError(
            f"Fresh validation population is still missing required columns: {missing}"
        )

    out.write_parquet(
        VALIDATION_POP_PARQUET,
        compression="zstd",
    )

    print("Saved refreshed validation population:", VALIDATION_POP_PARQUET)

    return out


if VALIDATION_POP_PARQUET.exists():
    cached_schema = pl.read_parquet_schema(VALIDATION_POP_PARQUET)
    cached_cols = set(cached_schema.keys())

    missing_cached = [
        c for c in REQUIRED_VALIDATION_FEATURES
        if c not in cached_cols
    ]

    if missing_cached:
        print(
            "Cached validation population is stale. "
            "Missing columns:",
            missing_cached,
        )

        # The previously cached population is still usable when the only
        # missing feature is the newly introduced T0 BIXS flag.
        # Reconstruct it from the already-cached T0 value rather than
        # rescanning the 20.9m-row V11 file unnecessarily.
        repairable = set(missing_cached).issubset(
            {"is_bi_excess_at_snapshot"}
        )

        if repairable:
            validation_pop = (
                pl.read_parquet(VALIDATION_POP_PARQUET)
                .with_columns(
                    (
                        pl.col("t0_bixs_v11").fill_null(0) > 0
                    )
                    .cast(pl.Int64)
                    .alias("is_bi_excess_at_snapshot")
                )
            )

            validation_pop.write_parquet(
                VALIDATION_POP_PARQUET,
                compression="zstd",
            )

            print(
                "Patched cached validation population with "
                "is_bi_excess_at_snapshot and re-saved it."
            )
        else:
            print(
                "Cache needs a full rebuild because additional model features "
                "are missing."
            )
            validation_pop = build_validation_population()

    else:
        validation_pop = pl.read_parquet(VALIDATION_POP_PARQUET)
        print("Using schema-complete cached validation population.")

else:
    validation_pop = build_validation_population()


# Final cache/model-feature audit.
missing_final = [
    c for c in REQUIRED_VALIDATION_FEATURES
    if c not in validation_pop.columns
]

if missing_final:
    raise RuntimeError(
        f"Validation population missing required model fields: {missing_final}"
    )

print("Validation population rows:", validation_pop.height)
print("Validation population columns:", len(validation_pop.columns))
print(
    "is_bi_excess_at_snapshot positives:",
    int(validation_pop["is_bi_excess_at_snapshot"].sum()),
)

display(
    validation_pop
    .group_by("model_band")
    .agg(
        [
            pl.len().alias("claims"),
            pl.col("actual_future_bixs")
            .sum()
            .alias("actual_future_bixs"),
            pl.col("is_bi_excess_at_snapshot")
            .sum()
            .alias("bixs_at_t0_claims"),
        ]
    )
    .sort("model_band")
)

In [ ]:
scorable_pop = validation_pop.filter(
    pl.col("model_band") != "UNSUPPORTED_EARLY"
)

unsupported_pop = validation_pop.filter(
    pl.col("model_band") == "UNSUPPORTED_EARLY"
)

SCORABLE_ACTUAL_REPORTED = float(
    scorable_pop["actual_future_bixs"].sum()
)

UNSUPPORTED_ACTUAL_REPORTED = float(
    unsupported_pop["actual_future_bixs"].sum()
)

print(
    f"Scorable reported target:   MYR {SCORABLE_ACTUAL_REPORTED/1e6:,.2f}m"
)
print(
    f"Unsupported DEV 0-3 target: MYR {UNSUPPORTED_ACTUAL_REPORTED/1e6:,.2f}m"
)
print(
    f"All reported claims:        MYR "
    f"{(SCORABLE_ACTUAL_REPORTED + UNSUPPORTED_ACTUAL_REPORTED)/1e6:,.2f}m"
)

reported_by_ay = (
    validation_pop
    .group_by("ACC_YEAR")
    .agg(
        [
            pl.col("actual_future_bixs")
            .sum()
            .alias("actual_reported_all"),

            pl.when(
                pl.col("model_band") != "UNSUPPORTED_EARLY"
            )
            .then(pl.col("actual_future_bixs"))
            .otherwise(0.0)
            .sum()
            .alias("actual_reported_scorable"),
        ]
    )
    .sort("ACC_YEAR")
)

display(reported_by_ay)

## 9. Define the frozen RF and XGBoost model artefacts

The six fitted model files are expected in their known Chapter 5 output folders: RF and
XGBoost models for DEV_QTR_4, DEV_QTR_8 and DEV_QTR_12. The notebook checks those
exact paths and stops if any fitted artefact is missing.

In [ ]:
CANONICAL_MODEL_ARTIFACTS = {
    "RF_DEV_QTR_4": (
        PROCESSED_DIR
        / "chapter5_outputs"
        / "section_5_7A_clean_rf_future_development"
        / "DEV_QTR_4"
        / "rf_future_development_DEV_QTR_4.joblib"
    ),
    "RF_DEV_QTR_8": (
        PROCESSED_DIR
        / "chapter5_outputs"
        / "section_5_7A_clean_rf_future_development"
        / "DEV_QTR_8"
        / "rf_future_development_DEV_QTR_8.joblib"
    ),
    "RF_DEV_QTR_12": (
        PROCESSED_DIR
        / "chapter5_outputs"
        / "section_5_7A_clean_rf_future_development"
        / "DEV_QTR_12"
        / "rf_future_development_DEV_QTR_12.joblib"
    ),
    "XGB_DEV_QTR_4": (
        PROCESSED_DIR
        / "chapter5_outputs"
        / "section_5_8A_clean_xgboost_future_development"
        / "DEV_QTR_4"
        / "xgb_future_development_DEV_QTR_4.joblib"
    ),
    "XGB_DEV_QTR_8": (
        PROCESSED_DIR
        / "chapter5_outputs"
        / "section_5_8A_clean_xgboost_future_development"
        / "DEV_QTR_8"
        / "xgb_future_development_DEV_QTR_8.joblib"
    ),
    "XGB_DEV_QTR_12": (
        PROCESSED_DIR
        / "chapter5_outputs"
        / "section_5_8A_clean_xgboost_future_development"
        / "DEV_QTR_12"
        / "xgb_future_development_DEV_QTR_12.joblib"
    ),
}


resolved_artifacts = {
    key: path
    for key, path in CANONICAL_MODEL_ARTIFACTS.items()
    if path.exists()
}

missing_artifacts = [
    f"{key}: {path}"
    for key, path in CANONICAL_MODEL_ARTIFACTS.items()
    if not path.exists()
]

if missing_artifacts:
    raise FileNotFoundError(
        "Frozen fitted model artefacts are missing:\n"
        + "\n".join(missing_artifacts)
    )

print("All six frozen fitted model artefacts confirmed.")


## 10. Load the frozen fitted models and preprocessing objects

The Random Forest files are fitted sklearn pipelines, while the XGBoost files also retain
their fitted preprocessing objects. These saved transformations must be reused exactly:
the later claim data are passed through the original preprocessing and fitted predictor,
with no refitting or recalibration.

In [ ]:
def load_raw_serialized_artifact(path: Path):
    suffix = path.suffix.lower()

    if suffix == ".joblib":
        if joblib is None:
            raise ImportError("joblib is not installed.")
        return joblib.load(path)

    with open(path, "rb") as f:
        return pickle.load(f)


loaded_models = {}
loaded_wrappers = {}
loaded_preprocessors = {}
loaded_model_metadata = {}

for key, path in resolved_artifacts.items():
    try:
        raw = load_raw_serialized_artifact(path)
        loaded_wrappers[key] = raw

        if hasattr(raw, "predict"):
            model = raw
            preprocessor = None
            metadata = {
                "wrapper_type": type(raw).__name__,
                "predictor_path": "root",
                "preprocessor_path": None,
            }

        elif isinstance(raw, dict):
            if "model" not in raw or not hasattr(raw["model"], "predict"):
                raise TypeError(
                    f"{key}: dictionary does not contain a fitted 'model' with predict(). "
                    f"Keys={list(raw.keys())}"
                )

            model = raw["model"]
            preprocessor = raw.get("preprocessor")

            if preprocessor is None or not hasattr(preprocessor, "transform"):
                raise TypeError(
                    f"{key}: XGBoost wrapper does not contain a fitted preprocessor "
                    "with transform()."
                )

            metadata = {
                "wrapper_type": "dict",
                "predictor_path": "root['model']",
                "preprocessor_path": "root['preprocessor']",
                "wrapper_keys": list(raw.keys()),
            }

        else:
            raise TypeError(
                f"{key}: unsupported saved object type {type(raw)}"
            )

        loaded_models[key] = model
        loaded_preprocessors[key] = preprocessor
        loaded_model_metadata[key] = metadata

        print("\n", key)
        print(" fitted object type:", type(model))
        print(" wrapper metadata:", metadata)

        if hasattr(model, "feature_names_in_"):
            print(" model feature_names_in_:", list(model.feature_names_in_))

        if preprocessor is not None:
            print(" preprocessor type:", type(preprocessor))
            if hasattr(preprocessor, "feature_names_in_"):
                print(
                    " preprocessor feature_names_in_:",
                    list(preprocessor.feature_names_in_)
                )

            feature_manifest = raw.get("feature_manifest")
            print(" feature_manifest type:", type(feature_manifest))
            if feature_manifest is not None:
                if isinstance(feature_manifest, dict):
                    print(" feature_manifest keys:", list(feature_manifest.keys()))
                elif isinstance(feature_manifest, (list, tuple)):
                    print(" feature_manifest length:", len(feature_manifest))

    except Exception as e:
        print(f"FAILED to load {key}: {e}")

In [ ]:
model_load_audit = pd.DataFrame(
    [
        {
            "model_key": key,
            "loaded": key in loaded_models,
            "fitted_object_type": (
                type(loaded_models[key]).__name__
                if key in loaded_models
                else None
            ),
            "wrapper_type": loaded_model_metadata.get(key, {}).get("wrapper_type"),
            "predictor_path": loaded_model_metadata.get(key, {}).get("predictor_path"),
            "preprocessor_path": loaded_model_metadata.get(key, {}).get("preprocessor_path"),
        }
        for key in sorted(resolved_artifacts)
    ]
)

display(model_load_audit)

missing_loaded = [
    key for key in CANONICAL_MODEL_ARTIFACTS
    if key not in loaded_models
]

if missing_loaded:
    raise RuntimeError(f"Frozen models failed to load: {missing_loaded}")

print("All six frozen fitted models loaded successfully.")

## 11. Score the frozen Random Forest and XGBoost models

Each supported claim is routed to the fitted model matching its maturity band. Mature
claims beyond DEV_QTR_15 are carried forward with zero predicted future development;
claims below DEV_QTR_4 are excluded from the reported-claim ML comparison. Predictions
are then summarised by accident year and at portfolio level.

In [ ]:
def dev_key_from_band(band):
    m = re.search(r"(4|8|12)", str(band))
    if not m:
        return None
    return int(m.group(1))


def model_key(model_name, band):
    d = dev_key_from_band(band)
    if d is None:
        return None
    return f"{model_name}_DEV_QTR_{d}"


def required_raw_features(key):
    model = loaded_models[key]
    preprocessor = loaded_preprocessors.get(key)
    wrapper = loaded_wrappers.get(key)

    if hasattr(model, "feature_names_in_"):
        return list(model.feature_names_in_)

    if preprocessor is not None and hasattr(preprocessor, "feature_names_in_"):
        return list(preprocessor.feature_names_in_)

    if isinstance(wrapper, dict):
        manifest = wrapper.get("feature_manifest")

        if isinstance(manifest, (list, tuple)) and all(
            isinstance(x, str) for x in manifest
        ):
            return list(manifest)

        if isinstance(manifest, dict):
            for candidate_key in [
                "features",
                "feature_names",
                "raw_features",
                "input_features",
                "model_features",
            ]:
                vals = manifest.get(candidate_key)
                if (
                    isinstance(vals, (list, tuple))
                    and vals
                    and all(isinstance(x, str) for x in vals)
                ):
                    return list(vals)

    raise RuntimeError(
        f"Could not recover the exact raw training feature list for {key}."
    )


def make_raw_X(key, frame: pl.DataFrame):
    required = required_raw_features(key)

    missing = [c for c in required if c not in frame.columns]
    if missing:
        raise KeyError(
            f"{key} requires columns missing from V11 T0 model data: {missing}"
        )

    return frame.select(required).to_pandas()


def predict_frozen(key, frame: pl.DataFrame):
    model = loaded_models[key]
    preprocessor = loaded_preprocessors.get(key)

    X_raw = make_raw_X(key, frame)

    if preprocessor is None:
        pred = model.predict(X_raw)
    else:
        X_transformed = preprocessor.transform(X_raw)
        pred = model.predict(X_transformed)

    pred = np.asarray(pred, dtype=float)

    if len(pred) != frame.height:
        raise RuntimeError(
            f"{key}: prediction length {len(pred)} != rows {frame.height}"
        )

    return pred

In [ ]:
feature_audit_rows = []

for key in sorted(loaded_models):
    features = required_raw_features(key)

    feature_audit_rows.append(
        {
            "model_key": key,
            "n_raw_features": len(features),
            "raw_features": " | ".join(features),
        }
    )

feature_audit = pd.DataFrame(feature_audit_rows)
display(feature_audit)

for key in sorted(loaded_models):
    features = required_raw_features(key)
    print(f"\n{key}: {len(features)} features")
    print(features)

In [ ]:
def score_model_family(model_name):
    pieces = []

    bands = sorted(
        set(scorable_pop["model_band"].unique().to_list())
    )

    for band in bands:
        band_df = scorable_pop.filter(pl.col("model_band") == band)

        print(
            f"{model_name}: band={band}, rows={band_df.height:,}"
        )

        if "MATURE" in str(band).upper():
            scored = (
                band_df
                .select(
                    claim_id
                    + [
                        "DEV_QTR",
                        "model_band",
                        "t0_bixs_v11",
                        "latest_bixs_v11",
                        "actual_future_bixs",
                    ]
                )
                .with_columns(
                    pl.lit(0.0)
                    .alias(f"{model_name.lower()}_predicted_future")
                )
            )
            pieces.append(scored)
            continue

        key = model_key(model_name, band)

        if key is None:
            raise RuntimeError(
                f"Could not map model band {band!r} to a frozen {model_name} model."
            )

        if key not in loaded_models:
            raise RuntimeError(
                f"Frozen model {key} is not loaded."
            )

        pred = predict_frozen(key, band_df)

        scored = (
            band_df
            .select(
                claim_id
                + [
                    "DEV_QTR",
                    "model_band",
                    "t0_bixs_v11",
                    "latest_bixs_v11",
                    "actual_future_bixs",
                ]
            )
            .with_columns(
                pl.Series(
                    f"{model_name.lower()}_predicted_future",
                    pred,
                )
            )
        )

        pieces.append(scored)

    if not pieces:
        return None

    return pl.concat(pieces)


print("\nScoring frozen Random Forest...")
rf_scored = score_model_family("RF")

print("\nScoring frozen XGBoost...")
xgb_scored = score_model_family("XGB")


In [ ]:
def summarise_scored(scored, pred_col):
    if scored is None:
        return None

    out = (
        scored
        .group_by("ACC_YEAR")
        .agg(
            [
                pl.col("actual_future_bixs")
                .sum()
                .alias("actual_reported_scorable"),

                pl.col(pred_col)
                .sum()
                .alias("predicted_reported"),

                pl.len().alias("claims_scored"),
            ]
        )
        .with_columns(
            [
                (
                    pl.col("predicted_reported")
                    - pl.col("actual_reported_scorable")
                ).alias("error_myr"),

                pl.when(pl.col("actual_reported_scorable") != 0)
                .then(
                    pl.col("predicted_reported")
                    / pl.col("actual_reported_scorable")
                )
                .otherwise(None)
                .alias("bias_ratio"),
            ]
        )
        .sort("ACC_YEAR")
    )

    return out


rf_by_ay = summarise_scored(
    rf_scored,
    "rf_predicted_future",
)

xgb_by_ay = summarise_scored(
    xgb_scored,
    "xgb_predicted_future",
)

if rf_by_ay is not None:
    print("\nRandom Forest out-of-time result")
    display(rf_by_ay)

    print(
        "RF portfolio predicted:",
        float(rf_scored["rf_predicted_future"].sum()),
    )
    print(
        "RF portfolio actual:",
        float(rf_scored["actual_future_bixs"].sum()),
    )

if xgb_by_ay is not None:
    print("\nXGBoost out-of-time result")
    display(xgb_by_ay)

    print(
        "XGB portfolio predicted:",
        float(xgb_scored["xgb_predicted_future"].sum()),
    )
    print(
        "XGB portfolio actual:",
        float(xgb_scored["actual_future_bixs"].sum()),
    )

## 12. Estimate pure IBNR using the historical frequency--severity method

The out-of-time pure IBNR allowance uses the same rolling three-comparator structure as
the historical Integrated Framework. Because the validation horizon is 13 quarters, the
comparator accident years must be sufficiently old for their complete 13-quarter outcomes
to have been observable by 2022 Q4; this gives lags of four, five and six years.

For target accident year $\mathrm{AY}$:

$$
\widehat{f}_{\mathrm{AY}}
=
\frac{
\sum \text{late positive-BI-Excess claims}
}{
\sum \text{reported claims at comparator snapshots}
}
$$

$$
\widehat{s}_{\mathrm{AY}}
=
\frac{
\sum \text{late-reported BI Excess amount}
}{
\sum \text{late positive-BI-Excess claims}
}
$$

and:

$$
\widehat{U}^{\mathrm{Pure\ IBNR}}_{\mathrm{AY}}
=
N^{\mathrm{Reported}}_{\mathrm{AY},T_0}
\widehat{f}_{\mathrm{AY}}
\widehat{s}_{\mathrm{AY}}
$$

Only information observable by 2022 Q4 is used to estimate this allowance. The later
extract is used afterwards to measure how much pure IBNR actually emerged.

In [ ]:
# Set the out-of-time dates and leakage-free comparator lags.
T0_LABEL = "2022 Q4"
T1_LABEL = "2026 Q1"

def quarter_label_to_index(label):
    year_text, quarter_text = str(label).strip().split()
    year = int(year_text)
    quarter = int(quarter_text.replace("Q", ""))
    return 4 * year + quarter

def quarter_index_to_label(index_value):
    index_value = int(index_value)
    year = (index_value - 1) // 4
    quarter = index_value - 4 * year
    return f"{year} Q{quarter}"

T0_INDEX = quarter_label_to_index(T0_LABEL)
T1_INDEX = quarter_label_to_index(T1_LABEL)
PURE_IBNR_HORIZON_QTRS = T1_INDEX - T0_INDEX

if PURE_IBNR_HORIZON_QTRS != 13:
    raise ValueError(
        f"Expected a 13-quarter out-of-time horizon, got "
        f"{PURE_IBNR_HORIZON_QTRS} quarters."
    )

MIN_COMPARATOR_LAG_YEARS = int(
    np.ceil(PURE_IBNR_HORIZON_QTRS / 4)
)
COMPARATOR_LAGS = list(
    range(
        MIN_COMPARATOR_LAG_YEARS,
        MIN_COMPARATOR_LAG_YEARS
        + PURE_IBNR_ROLLING_COMPARATOR_YEARS,
    )
)

historical_snapshot_indices = [
    T0_INDEX - 4 * lag
    for lag in COMPARATOR_LAGS
]
historical_outcome_indices = [
    snapshot_index + PURE_IBNR_HORIZON_QTRS
    for snapshot_index in historical_snapshot_indices
]

if max(historical_outcome_indices) > T0_INDEX:
    raise ValueError(
        "Comparator outcomes extend beyond 2022 Q4 and would introduce leakage."
    )

REQUIRED_POSITION_INDICES = sorted(
    set(historical_snapshot_indices + historical_outcome_indices)
)

print("Pure IBNR horizon:", PURE_IBNR_HORIZON_QTRS, "quarters")
print("Comparator lags:", COMPARATOR_LAGS)
print(
    pd.DataFrame({
        "comparator_lag_years": COMPARATOR_LAGS,
        "snapshot": [
            quarter_index_to_label(x)
            for x in historical_snapshot_indices
        ],
        "13q_outcome": [
            quarter_index_to_label(x)
            for x in historical_outcome_indices
        ],
    })
)

# Stream only the fields needed from the original master data.
MASTER_PATH = OLD_MASTER_PARQUET
parquet_file = pq.ParquetFile(MASTER_PATH)
available_columns = set(parquet_file.schema_arrow.names)

REQUIRED_MASTER_COLUMNS = [
    "SOURCE_FILE",
    "CLAIMS_KEY",
    "ACC_YEAR",
    "ACC_QTR",
    "DEV_QTR",
    "CLAIMS_CNT",
    "CUM_INC_LARGE",
]

missing_columns = sorted(
    set(REQUIRED_MASTER_COLUMNS).difference(available_columns)
)
if missing_columns:
    raise ValueError(
        f"Original master parquet is missing required fields: {missing_columns}"
    )

STREAM_BATCH_SIZE = 250_000
first_report_batches = []
position_batches = {
    valuation_index: []
    for valuation_index in REQUIRED_POSITION_INDICES
}

rows_scanned = 0

for batch_number, record_batch in enumerate(
    parquet_file.iter_batches(
        batch_size=STREAM_BATCH_SIZE,
        columns=REQUIRED_MASTER_COLUMNS,
    ),
    start=1,
):
    chunk = record_batch.to_pandas()
    rows_scanned += len(chunk)

    for col in [
        "ACC_YEAR",
        "ACC_QTR",
        "DEV_QTR",
        "CLAIMS_CNT",
        "CUM_INC_LARGE",
    ]:
        chunk[col] = pd.to_numeric(chunk[col], errors="coerce")

    chunk["CLAIMS_CNT"] = chunk["CLAIMS_CNT"].fillna(0.0)
    chunk["CUM_INC_LARGE"] = chunk["CUM_INC_LARGE"].fillna(0.0)

    chunk["ACCIDENT_QTR_INDEX"] = (
        4 * chunk["ACC_YEAR"].astype(int)
        + chunk["ACC_QTR"].astype(int)
    )
    chunk["VALUATION_QTR_INDEX"] = (
        chunk["ACCIDENT_QTR_INDEX"]
        + chunk["DEV_QTR"].astype(int)
    )

    report_rows = chunk.loc[
        chunk["CLAIMS_CNT"] > 0,
        [
            "SOURCE_FILE",
            "CLAIMS_KEY",
            "ACC_YEAR",
            "ACC_QTR",
            "VALUATION_QTR_INDEX",
        ],
    ]

    if not report_rows.empty:
        first_report_batch = (
            report_rows
            .groupby(
                [
                    "SOURCE_FILE",
                    "CLAIMS_KEY",
                    "ACC_YEAR",
                    "ACC_QTR",
                ],
                as_index=False,
            )
            .agg(first_report_index=("VALUATION_QTR_INDEX", "min"))
        )
        first_report_batches.append(first_report_batch)

    required_rows = chunk.loc[
        chunk["VALUATION_QTR_INDEX"].isin(REQUIRED_POSITION_INDICES),
        [
            "SOURCE_FILE",
            "CLAIMS_KEY",
            "ACC_YEAR",
            "ACC_QTR",
            "VALUATION_QTR_INDEX",
            "CUM_INC_LARGE",
            "CLAIMS_CNT",
        ],
    ]

    if not required_rows.empty:
        for valuation_index, date_rows in required_rows.groupby(
            "VALUATION_QTR_INDEX"
        ):
            position_batches[int(valuation_index)].append(
                date_rows.copy()
            )

    del chunk, report_rows, required_rows, record_batch

    if batch_number % 10 == 0:
        gc.collect()
        print(f"Scanned {rows_scanned:,} original-master rows...")

if not first_report_batches:
    raise ValueError("No reported claims were identified in the original master data.")

first_report = pd.concat(first_report_batches, ignore_index=True)
first_report = (
    first_report
    .groupby(
        [
            "SOURCE_FILE",
            "CLAIMS_KEY",
            "ACC_YEAR",
            "ACC_QTR",
        ],
        as_index=False,
    )
    .agg(first_report_index=("first_report_index", "min"))
)

del first_report_batches
gc.collect()

positions = {}

for valuation_index in REQUIRED_POSITION_INDICES:
    if not position_batches[valuation_index]:
        raise ValueError(
            f"No rows found for {quarter_index_to_label(valuation_index)}."
        )

    position_df = pd.concat(
        position_batches[valuation_index],
        ignore_index=True,
    )

    position_df = (
        position_df
        .sort_values(
            [
                "SOURCE_FILE",
                "CLAIMS_KEY",
                "VALUATION_QTR_INDEX",
            ]
        )
        .drop_duplicates(
            ["SOURCE_FILE", "CLAIMS_KEY"],
            keep="last",
        )
    )

    position_df = position_df.merge(
        first_report[
            [
                "SOURCE_FILE",
                "CLAIMS_KEY",
                "first_report_index",
            ]
        ],
        on=["SOURCE_FILE", "CLAIMS_KEY"],
        how="left",
        validate="one_to_one",
    )

    positions[valuation_index] = position_df

    print(
        quarter_index_to_label(valuation_index),
        "rows:",
        f"{len(position_df):,}",
        "| BIXS:",
        f"{position_df['CUM_INC_LARGE'].sum():,.2f}",
    )

del position_batches
gc.collect()


In [ ]:
KEY_COLUMNS = ["SOURCE_FILE", "CLAIMS_KEY"]

def cohort_metrics(accident_year, snapshot_index, outcome_index):
    snapshot = positions[snapshot_index].loc[
        positions[snapshot_index]["ACC_YEAR"].eq(accident_year)
    ].copy()

    outcome = positions[outcome_index].loc[
        positions[outcome_index]["ACC_YEAR"].eq(accident_year)
    ].copy()

    cohort_reports = first_report.loc[
        first_report["ACC_YEAR"].eq(accident_year)
    ].copy()

    reported_keys = cohort_reports.loc[
        cohort_reports["first_report_index"] <= snapshot_index,
        KEY_COLUMNS,
    ].drop_duplicates()

    reported_claim_count = len(reported_keys)

    outcome_with_report = outcome.merge(
        cohort_reports[
            KEY_COLUMNS + ["first_report_index"]
        ],
        on=KEY_COLUMNS,
        how="left",
        validate="one_to_one",
        suffixes=("", "_report"),
    )

    pure_ibnr_outcome = outcome_with_report.loc[
        (outcome_with_report["first_report_index"] > snapshot_index)
        & (outcome_with_report["first_report_index"] <= outcome_index)
        & (outcome_with_report["CUM_INC_LARGE"] > 0)
    ]

    return {
        "reported_claim_count_at_snapshot": int(reported_claim_count),
        "pure_ibnr_positive_bixs_claim_count": int(
            len(pure_ibnr_outcome)
        ),
        "pure_ibnr_bixs_amount": float(
            pure_ibnr_outcome["CUM_INC_LARGE"].sum()
        ),
    }


# Accident years supported by the fitted claim-level maturity bands.
# This definition was present in the original v7 notebook and must be
# created before the pure-IBNR comparator loop below.
supported_ays = (
    validation_pop
    .filter(pl.col("model_band") != "UNSUPPORTED_EARLY")
    .select("ACC_YEAR")
    .unique()
)

supported_ay_values = sorted(
    int(x)
    for x in supported_ays["ACC_YEAR"].to_list()
)

minimum_data_ay = int(first_report["ACC_YEAR"].min())

comparator_rows = []

for target_ay in supported_ay_values:
    for lag in COMPARATOR_LAGS:
        comparator_ay = target_ay - lag

        if comparator_ay < minimum_data_ay:
            continue

        snapshot_index = T0_INDEX - 4 * lag
        outcome_index = (
            snapshot_index + PURE_IBNR_HORIZON_QTRS
        )

        if outcome_index > T0_INDEX:
            raise ValueError(
                "Comparator outcome occurs after T0 and introduces leakage."
            )

        metrics = cohort_metrics(
            comparator_ay,
            snapshot_index,
            outcome_index,
        )

        comparator_rows.append({
            "target_accident_year": target_ay,
            "comparator_lag_years": lag,
            "comparator_accident_year": comparator_ay,
            "snapshot_label": quarter_index_to_label(snapshot_index),
            "outcome_label": quarter_index_to_label(outcome_index),
            **metrics,
        })

comparator_detail = pd.DataFrame(comparator_rows)

comparator_detail.to_csv(
    OUT_DIR / "pure_ibnr_13q_comparator_detail.csv",
    index=False,
)

display(comparator_detail.tail(15))


def target_reported_claim_count(accident_year):
    target_reports = first_report.loc[
        first_report["ACC_YEAR"].eq(accident_year)
        & (first_report["first_report_index"] <= T0_INDEX),
        KEY_COLUMNS,
    ].drop_duplicates()

    return int(len(target_reports))


pure_ibnr_rows = []

for target_ay in supported_ay_values:
    comparators = comparator_detail.loc[
        comparator_detail["target_accident_year"].eq(target_ay)
    ].copy()

    comparator_exposure = float(
        comparators["reported_claim_count_at_snapshot"].sum()
    )
    comparator_pure_count = float(
        comparators["pure_ibnr_positive_bixs_claim_count"].sum()
    )
    comparator_pure_amount = float(
        comparators["pure_ibnr_bixs_amount"].sum()
    )

    frequency = (
        comparator_pure_count / comparator_exposure
        if comparator_exposure > 0
        else 0.0
    )

    severity = (
        comparator_pure_amount / comparator_pure_count
        if comparator_pure_count > 0
        else 0.0
    )

    amount_per_reported_claim = (
        comparator_pure_amount / comparator_exposure
        if comparator_exposure > 0
        else 0.0
    )

    target_exposure = target_reported_claim_count(target_ay)

    estimated_amount = (
        target_exposure
        * frequency
        * severity
    )

    # The algebraically equivalent form is retained as an audit check.
    estimated_amount_direct = (
        target_exposure
        * amount_per_reported_claim
    )

    if abs(estimated_amount - estimated_amount_direct) > 1e-6:
        raise RuntimeError(
            "Frequency-severity and direct amount-per-exposure "
            "pure-IBNR calculations do not reconcile."
        )

    pure_ibnr_rows.append({
        "ACC_YEAR": target_ay,
        "T0": T0_LABEL,
        "T1": T1_LABEL,
        "comparator_accident_years": ",".join(
            str(int(v))
            for v in comparators["comparator_accident_year"].tolist()
        ),
        "comparator_year_count": int(len(comparators)),
        "target_reported_claim_count_at_T0": target_exposure,
        "comparator_reported_claim_exposure": comparator_exposure,
        "comparator_pure_ibnr_positive_claim_count": comparator_pure_count,
        "comparator_pure_ibnr_amount": comparator_pure_amount,
        "estimated_pure_ibnr_frequency": frequency,
        "estimated_pure_ibnr_severity": severity,
        "estimated_pure_ibnr_amount_per_reported_claim": (
            amount_per_reported_claim
        ),
        "estimated_pure_ibnr_bixs_amount": estimated_amount,
    })

pure_ibnr_by_ay = pd.DataFrame(pure_ibnr_rows)

if emergence_ay is None:
    raise RuntimeError(
        "The later emergence-by-AY file is required for validation."
    )

actual_pure_by_ay = (
    emergence_ay
    .select(
        [
            "ACC_YEAR",
            pl.col("PURE_IBNR_PROXY").alias(
                "observed_pure_ibnr_bixs_amount_T1"
            ),
        ]
    )
    .to_pandas()
)

pure_ibnr_by_ay = pure_ibnr_by_ay.merge(
    actual_pure_by_ay,
    on="ACC_YEAR",
    how="left",
    validate="one_to_one",
)

pure_ibnr_by_ay["pure_ibnr_amount_error"] = (
    pure_ibnr_by_ay["estimated_pure_ibnr_bixs_amount"]
    - pure_ibnr_by_ay["observed_pure_ibnr_bixs_amount_T1"]
)

pure_ibnr_by_ay.to_csv(
    OUT_DIR / "pure_ibnr_13q_frequency_severity_by_AY.csv",
    index=False,
)

display(pure_ibnr_by_ay)


## 13. Add pure IBNR to the reported-claim predictions

The same accident-year pure IBNR allowance is added to both RF and XGBoost so that the
difference between the two Integrated Framework results comes only from their reported-
claim predictions. The combined estimates are then compared with the observed supported
portfolio, separating reported-claim performance from pure IBNR performance.

In [ ]:
actual_hybrid_by_ay = (
    emergence_ay
    .join(supported_ays, on="ACC_YEAR", how="inner")
    .select(
        [
            "ACC_YEAR",
            pl.col("BIXS_AT_2022Q4").alias("actual_existing_bixs_dev"),
            pl.col("REPORTED_BIC_TO_BIXS").alias("actual_reported_crossing"),
            pl.col("PURE_IBNR_PROXY").alias("actual_pure_ibnr"),
            (
                pl.col("BIXS_AT_2022Q4")
                + pl.col("REPORTED_BIC_TO_BIXS")
                + pl.col("PURE_IBNR_PROXY")
            ).alias("actual_supported_total"),
        ]
    )
    .sort("ACC_YEAR")
)

pure_estimate_pl = pl.from_pandas(
    pure_ibnr_by_ay[
        [
            "ACC_YEAR",
            "estimated_pure_ibnr_bixs_amount",
        ]
    ]
).with_columns(
    pl.col("ACC_YEAR").cast(pl.Int64)
)

ACTUAL_SUPPORTED_TOTAL = float(
    actual_hybrid_by_ay["actual_supported_total"].sum()
)

ACTUAL_SUPPORTED_PURE_IBNR = float(
    actual_hybrid_by_ay["actual_pure_ibnr"].sum()
)

def integrated_summary_by_ay(
    reported_summary,
    prediction_label,
):
    if reported_summary is None:
        return None

    return (
        actual_hybrid_by_ay
        .join(
            reported_summary.select(
                [
                    "ACC_YEAR",
                    pl.col("predicted_reported").alias(
                        f"{prediction_label}_reported_prediction"
                    ),
                ]
            ),
            on="ACC_YEAR",
            how="left",
        )
        .join(
            pure_estimate_pl,
            on="ACC_YEAR",
            how="left",
        )
        .with_columns(
            pl.col("estimated_pure_ibnr_bixs_amount")
            .fill_null(0.0)
        )
        .with_columns(
            [
                pl.col("estimated_pure_ibnr_bixs_amount")
                .alias(
                    f"{prediction_label}_predicted_pure_ibnr"
                ),

                (
                    pl.col(f"{prediction_label}_reported_prediction")
                    + pl.col("estimated_pure_ibnr_bixs_amount")
                ).alias(
                    f"{prediction_label}_hybrid_predicted"
                ),
            ]
        )
        .with_columns(
            (
                pl.col(f"{prediction_label}_hybrid_predicted")
                - pl.col("actual_supported_total")
            ).alias(f"{prediction_label}_hybrid_error")
        )
        .sort("ACC_YEAR")
    )


rf_hybrid_by_ay = integrated_summary_by_ay(
    rf_by_ay,
    "rf",
)

xgb_hybrid_by_ay = integrated_summary_by_ay(
    xgb_by_ay,
    "xgb",
)

RF_HYBRID_TOTAL = (
    float(rf_hybrid_by_ay["rf_hybrid_predicted"].sum())
    if rf_hybrid_by_ay is not None
    else None
)

XGB_HYBRID_TOTAL = (
    float(xgb_hybrid_by_ay["xgb_hybrid_predicted"].sum())
    if xgb_hybrid_by_ay is not None
    else None
)

ESTIMATED_PURE_IBNR_TOTAL = float(
    pure_ibnr_by_ay["estimated_pure_ibnr_bixs_amount"].sum()
)

print(
    f"Estimated pure IBNR on supported AY scope: "
    f"MYR {ESTIMATED_PURE_IBNR_TOTAL/1e6:,.2f}m"
)
print(
    f"Observed pure IBNR on supported AY scope:  "
    f"MYR {ACTUAL_SUPPORTED_PURE_IBNR/1e6:,.2f}m"
)
print(
    "RF Integrated Framework prediction:",
    None if RF_HYBRID_TOTAL is None
    else f"MYR {RF_HYBRID_TOTAL/1e6:,.2f}m",
)
print(
    "XGB Integrated Framework prediction:",
    None if XGB_HYBRID_TOTAL is None
    else f"MYR {XGB_HYBRID_TOTAL/1e6:,.2f}m",
)
print(
    f"Supported actual total: MYR {ACTUAL_SUPPORTED_TOTAL/1e6:,.2f}m"
)


In [ ]:
if rf_hybrid_by_ay is not None:
    display(
        rf_hybrid_by_ay.select(
            [
                "ACC_YEAR",
                "actual_supported_total",
                "actual_pure_ibnr",
                "rf_reported_prediction",
                "rf_predicted_pure_ibnr",
                "rf_hybrid_predicted",
                "rf_hybrid_error",
            ]
        )
    )

if xgb_hybrid_by_ay is not None:
    display(
        xgb_hybrid_by_ay.select(
            [
                "ACC_YEAR",
                "actual_supported_total",
                "actual_pure_ibnr",
                "xgb_reported_prediction",
                "xgb_predicted_pure_ibnr",
                "xgb_hybrid_predicted",
                "xgb_hybrid_error",
            ]
        )
    )

pure_ibnr_audit = pd.DataFrame([{
    "T0": T0_LABEL,
    "T1": T1_LABEL,
    "horizon_quarters": PURE_IBNR_HORIZON_QTRS,
    "comparator_lags_years": ",".join(
        str(x) for x in COMPARATOR_LAGS
    ),
    "estimated_pure_ibnr_total": ESTIMATED_PURE_IBNR_TOTAL,
    "observed_pure_ibnr_total": ACTUAL_SUPPORTED_PURE_IBNR,
    "pure_ibnr_error": (
        ESTIMATED_PURE_IBNR_TOTAL
        - ACTUAL_SUPPORTED_PURE_IBNR
    ),
}])

pure_ibnr_audit.to_csv(
    OUT_DIR / "pure_ibnr_13q_frequency_severity_audit.csv",
    index=False,
)

display(pure_ibnr_audit)


## 14. Rebuild the carried-forward incurred BI Excess triangle

For Chain Ladder, cumulative incurred amounts must remain in the triangle even when a
claim has no later physical row. The code therefore converts each claim history into
changes in cumulative BI Excess, aggregates those changes by origin and development
quarter, and cumulatively sums them to carry the last observed incurred amount forward.
Only data available by the original 2022 Q4 cut are used.

In [ ]:
CF_TRIANGLE_PARQUET = (
    BASE_VALIDATION_DIR / "old_2022q4_carried_forward_incurred_bixs_triangle.parquet"
)

def build_carried_forward_triangle():
    if CF_TRIANGLE_PARQUET.exists():
        print("Using cached carried-forward triangle:", CF_TRIANGLE_PARQUET)
        return pl.read_parquet(CF_TRIANGLE_PARQUET)

    origin_parts = []

    claim_hist_id = [
        "SOURCE_FILE",
        "CLAIMS_KEY",
        "ACC_YEAR",
        "ACC_QTR",
    ]

    for year in range(2010, 2023):
        print(f"Building carried-forward triangle AY {year}...")

        df = (
            pl.scan_parquet(OLD_MASTER_PARQUET)
            .filter(pl.col("ACC_YEAR") == year)
            .select(
                claim_hist_id
                + [
                    "DEV_QTR",
                    "CUM_INC_LARGE",
                ]
            )
            .collect()
            .sort(claim_hist_id + ["DEV_QTR"])
            .with_columns(
                pl.col("CUM_INC_LARGE")
                .fill_null(0)
                .alias("_CURRENT_BIXS")
            )
            .with_columns(
                (
                    pl.col("_CURRENT_BIXS")
                    - pl.col("_CURRENT_BIXS")
                    .shift(1)
                    .over(claim_hist_id)
                    .fill_null(0)
                ).alias("_BIXS_CHANGE")
            )
        )

        increments = (
            df
            .group_by(["ACC_YEAR", "ACC_QTR", "DEV_QTR"])
            .agg(
                pl.col("_BIXS_CHANGE")
                .sum()
                .alias("incremental_bixs")
            )
        )

        for qtr in [1, 2, 3, 4]:
            max_dev = old_diagonal_map[(year, qtr)]

            grid = pl.DataFrame(
                {
                    "ACC_YEAR": [year] * (max_dev + 1),
                    "ACC_QTR": [qtr] * (max_dev + 1),
                    "DEV_QTR": list(range(max_dev + 1)),
                }
            )

            q = (
                grid
                .join(
                    increments.filter(pl.col("ACC_QTR") == qtr),
                    on=["ACC_YEAR", "ACC_QTR", "DEV_QTR"],
                    how="left",
                )
                .with_columns(
                    pl.col("incremental_bixs").fill_null(0)
                )
                .sort("DEV_QTR")
                .with_columns(
                    pl.col("incremental_bixs")
                    .cum_sum()
                    .alias("cum_bixs_cf")
                )
            )

            origin_parts.append(q)

        del df, increments
        gc.collect()

    tri = pl.concat(origin_parts)

    tri.write_parquet(
        CF_TRIANGLE_PARQUET,
        compression="zstd",
    )

    print("Saved carried-forward triangle:", CF_TRIANGLE_PARQUET)
    return tri


cf_triangle = build_carried_forward_triangle()

print("Triangle rows:", cf_triangle.height)
print(
    "Development range:",
    cf_triangle["DEV_QTR"].min(),
    "to",
    cf_triangle["DEV_QTR"].max(),
)

display(cf_triangle.head(12))

## 15. Fit the incurred Chain Ladder and project 13 quarters

Volume-weighted development factors are fitted to the carried-forward triangle using only
the original 2022 Q4 information. They are then applied to the later-extract restated 2022
Q4 starting amounts to produce an exact 13-quarter forecast through 2026 Q1, together
with a terminal/ultimate-style sensitivity. Any part of the 13-quarter horizon extending
beyond the oldest observed development age is explicitly counted as an unobserved tail.

In [ ]:
HORIZON_QUARTERS = 13

def fit_volume_weighted_ldfs(tri):
    max_dev = int(tri["DEV_QTR"].max())
    rows = []

    for d in range(max_dev):
        pair = (
            tri
            .filter(pl.col("DEV_QTR").is_in([d, d + 1]))
            .pivot(
                values="cum_bixs_cf",
                index=["ACC_YEAR", "ACC_QTR"],
                on="DEV_QTR",
                aggregate_function="first",
            )
        )

        c0 = str(d)
        c1 = str(d + 1)

        if c0 not in pair.columns or c1 not in pair.columns:
            continue

        valid = pair.filter(
            pl.col(c0).is_not_null()
            & pl.col(c1).is_not_null()
        )

        denominator = float(valid[c0].sum())
        numerator = float(valid[c1].sum())

        ldf = numerator / denominator if denominator != 0 else 1.0

        rows.append(
            {
                "DEV_QTR": d,
                "LDF": ldf,
                "denominator": denominator,
                "numerator": numerator,
                "origin_pairs": valid.height,
            }
        )

    return pl.DataFrame(rows)


cl_cf_factors = fit_volume_weighted_ldfs(cf_triangle)
display(cl_cf_factors)

cl_cf_factors.write_csv(
    OUT_DIR / "incurred_bixs_chain_ladder_2022q4_carried_forward_factors.csv"
)

LDF_MAP = {
    int(r["DEV_QTR"]): float(r["LDF"])
    for r in cl_cf_factors.to_dicts()
}

MAX_FACTOR_START = max(LDF_MAP)
MAX_OBSERVED_DEV = MAX_FACTOR_START + 1

print("Maximum observed development:", MAX_OBSERVED_DEV)

In [ ]:
# V11-restated starting triangle at the exact 2022 Q4 diagonal.
v11_t0_origin = (
    t0
    .group_by(["ACC_YEAR", "ACC_QTR"])
    .agg(
        pl.col("CUM_INC_LARGE")
        .fill_null(0)
        .sum()
        .alias("v11_t0_bixs")
    )
)

diag_for_join = old_diagonal.rename({"T0_DEV_QTR": "t0_dev_qtr"})

v11_t0_origin = (
    v11_t0_origin
    .join(
        diag_for_join,
        on=["ACC_YEAR", "ACC_QTR"],
        how="inner",
    )
    .sort(["ACC_YEAR", "ACC_QTR"])
)

def factor_product(start_dev, number_of_quarters=None):
    start_dev = int(start_dev)

    if number_of_quarters is None:
        stop_exclusive = MAX_OBSERVED_DEV
    else:
        stop_exclusive = min(
            start_dev + int(number_of_quarters),
            MAX_OBSERVED_DEV,
        )

    f = 1.0
    for k in range(start_dev, stop_exclusive):
        f *= LDF_MAP.get(k, 1.0)

    return f


forecast_rows = []

for r in v11_t0_origin.to_dicts():
    d = int(r["t0_dev_qtr"])
    t0_amt = float(r["v11_t0_bixs"] or 0.0)

    factor_13q = factor_product(d, HORIZON_QUARTERS)
    factor_terminal = factor_product(d, None)

    forecast_rows.append(
        {
            "ACC_YEAR": int(r["ACC_YEAR"]),
            "ACC_QTR": int(r["ACC_QTR"]),
            "t0_dev_qtr": d,
            "v11_t0_bixs": t0_amt,
            "cl_13q_factor": factor_13q,
            "cl_13q_projected_bixs": t0_amt * factor_13q,
            "cl_13q_predicted_future": t0_amt * (factor_13q - 1.0),
            "cl_terminal_factor": factor_terminal,
            "cl_terminal_projected_bixs": t0_amt * factor_terminal,
            "cl_terminal_predicted_future": t0_amt * (factor_terminal - 1.0),
            "unobserved_tail_quarters_in_13q": max(
                0,
                (d + HORIZON_QUARTERS) - MAX_OBSERVED_DEV,
            ),
        }
    )

cl_origin_forecast = pl.DataFrame(forecast_rows)

cl_13q_by_ay = (
    cl_origin_forecast
    .group_by("ACC_YEAR")
    .agg(
        [
            pl.col("v11_t0_bixs").sum(),
            pl.col("cl_13q_projected_bixs").sum(),
            pl.col("cl_13q_predicted_future").sum(),
            pl.col("cl_terminal_projected_bixs").sum(),
            pl.col("cl_terminal_predicted_future").sum(),
            pl.col("unobserved_tail_quarters_in_13q").max(),
        ]
    )
    .sort("ACC_YEAR")
)

actual_total_by_ay = (
    bridge
    .select(
        [
            "ACC_YEAR",
            pl.col("bixs_post_2022_emergence")
            .alias("actual_2022q4_to_2026q1"),
        ]
    )
    .sort("ACC_YEAR")
)

cl_horizon_validation = (
    actual_total_by_ay
    .join(cl_13q_by_ay, on="ACC_YEAR", how="left")
    .with_columns(
        [
            (
                pl.col("cl_13q_predicted_future")
                - pl.col("actual_2022q4_to_2026q1")
            ).alias("cl_13q_error"),

            (
                pl.col("cl_terminal_predicted_future")
                - pl.col("actual_2022q4_to_2026q1")
            ).alias("cl_terminal_error"),
        ]
    )
)

display(cl_horizon_validation)

cl_origin_forecast.write_csv(
    OUT_DIR / "incurred_cl_2022q4_to_2026q1_by_accident_quarter.csv"
)

cl_horizon_validation.write_csv(
    OUT_DIR / "incurred_cl_2022q4_to_2026q1_by_AY.csv"
)

CL_13Q_TOTAL = float(
    cl_horizon_validation["cl_13q_predicted_future"].sum()
)

CL_TERMINAL_TOTAL = float(
    cl_horizon_validation["cl_terminal_predicted_future"].sum()
)

print(f"13-quarter CL predicted future: MYR {CL_13Q_TOTAL/1e6:,.2f}m")
print(f"Terminal CL predicted future:   MYR {CL_TERMINAL_TOTAL/1e6:,.2f}m")
print(f"Actual 13-quarter emergence:    MYR {ACTUAL_TOTAL_POST_2022/1e6:,.2f}m")
print(f"13-quarter CL bias ratio:       {CL_13Q_TOTAL/ACTUAL_TOTAL_POST_2022:.3f}")
print(f"Terminal CL bias ratio:         {CL_TERMINAL_TOTAL/ACTUAL_TOTAL_POST_2022:.3f}")

## 16. Compare the out-of-time results on the correct scopes

The final portfolio table keeps three bases separate: RF/XGBoost reported-claim
development, the Integrated Framework after adding pure IBNR, and full-portfolio
Chain Ladder. This avoids attributing unsupported DEV_QTR_0--3 exposure to the
claim-level models and makes clear which observed total belongs to each comparison.

In [ ]:
final_rows = []

def comparison_row(method, target, predicted, actual, scope):
    predicted = float(predicted)
    actual = float(actual)

    return {
        "Method": method,
        "Target": target,
        "Predicted_MYR_m": predicted / 1e6,
        "Actual_MYR_m": actual / 1e6,
        "Error_MYR_m": (predicted - actual) / 1e6,
        "Bias_Ratio": predicted / actual if actual != 0 else np.nan,
        "Scope": scope,
    }


if rf_scored is not None:
    final_rows.append(
        comparison_row(
            "Random Forest",
            "Reported-claim future BIXS",
            rf_scored["rf_predicted_future"].sum(),
            rf_scored["actual_future_bixs"].sum(),
            "Supported valuation-diagonal claims; DEV_QTR 0-3 excluded",
        )
    )

if xgb_scored is not None:
    final_rows.append(
        comparison_row(
            "XGBoost",
            "Reported-claim future BIXS",
            xgb_scored["xgb_predicted_future"].sum(),
            xgb_scored["actual_future_bixs"].sum(),
            "Supported valuation-diagonal claims; DEV_QTR 0-3 excluded",
        )
    )

if RF_HYBRID_TOTAL is not None:
    final_rows.append(
        comparison_row(
            "Integrated Framework — RF",
            "Reported + rolling frequency-severity pure IBNR",
            RF_HYBRID_TOTAL,
            ACTUAL_SUPPORTED_TOTAL,
            "Supported valuation-diagonal portfolio; DEV_QTR 0-3 excluded",
        )
    )

if XGB_HYBRID_TOTAL is not None:
    final_rows.append(
        comparison_row(
            "Integrated Framework — XGBoost",
            "Reported + rolling frequency-severity pure IBNR",
            XGB_HYBRID_TOTAL,
            ACTUAL_SUPPORTED_TOTAL,
            "Same rolling pure-IBNR allowance applied to XGBoost",
        )
    )

final_rows.append(
    comparison_row(
        "Incurred Chain Ladder — 13 quarter",
        "Total BIXS emergence through 2026 Q1",
        CL_13Q_TOTAL,
        ACTUAL_TOTAL_POST_2022,
        "Full 2010-2022 portfolio; V11-restated 2022Q4 starting basis",
    )
)

final_rows.append(
    comparison_row(
        "Incurred Chain Ladder — terminal",
        "Terminal/ultimate-style BIXS development",
        CL_TERMINAL_TOTAL,
        ACTUAL_TOTAL_POST_2022,
        "Sensitivity: compared with observed V11 position through 2026Q1",
    )
)

final_comparison = pd.DataFrame(final_rows)

display(final_comparison)

final_comparison.to_csv(
    OUT_DIR / "final_out_of_time_model_comparison_portfolio.csv",
    index=False,
)

print(
    "Saved:",
    OUT_DIR / "final_out_of_time_model_comparison_portfolio.csv"
)

## 17. Build the final accident-year comparison

The reported-claim, pure IBNR, Integrated Framework and Chain Ladder results are joined
by accident year. This provides the detailed table needed to identify offsetting positive
and negative errors that may be hidden by the portfolio totals.

In [ ]:
final_ay = (
    reported_by_ay
    .join(
        bridge.select(
            [
                "ACC_YEAR",
                pl.col("bixs_post_2022_emergence")
                .alias("actual_total_13q"),
            ]
        ),
        on="ACC_YEAR",
        how="left",
    )
)

if rf_by_ay is not None:
    final_ay = final_ay.join(
        rf_by_ay.select(
            [
                "ACC_YEAR",
                pl.col("predicted_reported").alias("rf_reported_prediction"),
            ]
        ),
        on="ACC_YEAR",
        how="left",
    )

if xgb_by_ay is not None:
    final_ay = final_ay.join(
        xgb_by_ay.select(
            [
                "ACC_YEAR",
                pl.col("predicted_reported").alias("xgb_reported_prediction"),
            ]
        ),
        on="ACC_YEAR",
        how="left",
    )

if rf_hybrid_by_ay is not None:
    final_ay = final_ay.join(
        rf_hybrid_by_ay.select(
            [
                "ACC_YEAR",
                "actual_pure_ibnr",
                "actual_supported_total",
                "rf_predicted_pure_ibnr",
                "rf_hybrid_predicted",
            ]
        ),
        on="ACC_YEAR",
        how="left",
    )

if xgb_hybrid_by_ay is not None:
    final_ay = final_ay.join(
        xgb_hybrid_by_ay.select(
            [
                "ACC_YEAR",
                "xgb_predicted_pure_ibnr",
                "xgb_hybrid_predicted",
            ]
        ),
        on="ACC_YEAR",
        how="left",
    )

final_ay = final_ay.join(
    cl_horizon_validation.select(
        [
            "ACC_YEAR",
            "cl_13q_predicted_future",
            "cl_terminal_predicted_future",
        ]
    ),
    on="ACC_YEAR",
    how="left",
).sort("ACC_YEAR")

display(final_ay)

final_ay.write_csv(
    OUT_DIR / "final_out_of_time_model_comparison_by_AY.csv"
)

print(
    "Saved:",
    OUT_DIR / "final_out_of_time_model_comparison_by_AY.csv"
)